# Pattern 6: Gateway + JWT + Cedar Policy (Multi-Tenant)

Combine JWT authentication with Cedar authorization. JWT validates WHO the caller is,
Cedar decides WHAT they can do. This is the pattern for multi-tenant applications where
different users/departments have different access levels.

**Two-layer auth model:**
- **Layer 1 — JWT (Cognito):** Validates identity. Is this a real, authenticated user?
- **Layer 2 — Cedar:** Evaluates authorization. Is this user allowed to invoke this gateway/target?

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the
  sample documents, and creates the KB execution role. (The setup cell here re-runs it
  idempotently, and additionally creates the **Gateway role**.)
- IAM permissions for Bedrock, AgentCore (`bedrock-agentcore-control` — Gateway, Policy
  Engine, Cedar policies), Cognito, S3, and IAM.

## Architecture

```
Agent ──► JWT Token ──► Gateway ──► Cedar Policy Engine ──► KB Target ──► Managed KB
  │                        │              │
  │                        │              └── permit/deny based on principal + resource
  │                        └── Validates JWT via Cognito OIDC discovery
  └── Cognito User Pool
```

In [ ]:
import boto3
import time
import json
import util   # util.py in this folder — shared bucket + upload + roles

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + KB execution role as Pattern 1 (idempotent),
# then create the Gateway role the AgentCore Gateway assumes to retrieve.
info = util.setup(
    bucket_name=S3_BUCKET,
    prefix=S3_PREFIX,
    metadata=util.SAMPLE_FILE_METADATA,
    region_name=REGION,
)
ROLE_ARN    = info["role_arn"]
S3_BUCKET   = info["bucket"]
S3_PREFIX   = info["prefix"]
GW_ROLE_ARN = util.create_gateway_role(region_name=REGION)

# Clients
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
ac = session.client("bedrock-agentcore-control", region_name=REGION)
cognito = session.client("cognito-idp", region_name=REGION)
ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
S3_ACCOUNT = ACCOUNT_ID

print(f"boto3 {boto3.__version__}")
print(f"KB role:      {ROLE_ARN}")
print(f"Gateway role: {GW_ROLE_ARN}")


In [ ]:
# Step 1: Create Cognito User Pool + test user + Policy Engine
pool = cognito.create_user_pool(
    PoolName=f"p6-jwt-cedar-{int(time.time())}",
    Policies={"PasswordPolicy": {
        "MinimumLength": 8, "RequireUppercase": False,
        "RequireLowercase": False, "RequireNumbers": False, "RequireSymbols": False
    }},
    Schema=[{
        "Name": "department", "AttributeDataType": "String",
        "Mutable": True, "Required": False,
        "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "256"}
    }]
)
pool_id = pool["UserPool"]["Id"]

client_resp = cognito.create_user_pool_client(
    UserPoolId=pool_id, ClientName="p6-client",
    ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
    GenerateSecret=False
)
client_id = client_resp["UserPoolClient"]["ClientId"]
discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{pool_id}/.well-known/openid-configuration"
print(f"Cognito Pool: {pool_id}")
print(f"App Client:   {client_id}")

# Test user (department claim available for the multi-tenant discussion below)
cognito.admin_create_user(
    UserPoolId=pool_id, Username="testuser",
    UserAttributes=[
        {"Name": "custom:department", "Value": "engineering"},
        {"Name": "email", "Value": "test@example.com"}
    ],
    MessageAction="SUPPRESS"
)
cognito.admin_set_user_password(
    UserPoolId=pool_id, Username="testuser",
    Password="TestPass1", Permanent=True
)
print("Test user created: testuser (department=engineering)")

# Policy Engine
pe = ac.create_policy_engine(name=f"p6pe{int(time.time())}")
pe_id = pe["policyEngineId"]
pe_arn = pe["policyEngineArn"]
for _ in range(12):
    if ac.get_policy_engine(policyEngineId=pe_id)["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Policy Engine: {pe_id} ACTIVE")

In [ ]:
# Step 2: Create KB + Data Source + Ingest (same as Pattern 1)
response = cp.create_knowledge_base(
    name=f"p6-jwt-cedar-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {"inclusionPrefixes": [S3_PREFIX]},
                "deletionProtectionConfiguration": {"enableDeletionProtection": False}
            },
            "deletionProtectionConfiguration": {"deletionProtectionStatus": "DISABLED"}
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)
print(f"Ingestion: {job['status']}")

In [ ]:
# Step 3: Create Gateway with CUSTOM_JWT + Cedar (LOG_ONLY)
# JWT (Layer 1) validates WHO the caller is; Cedar (Layer 2) decides WHAT they can do.
#
# allowedAudience vs allowedClients: the authorizer verifies EVERY field you
# configure, but Cognito splits these claims across its two tokens — the ACCESS
# token carries `client_id` (and NO `aud`), while the ID token carries `aud`
# (and NO `client_id`). Requiring BOTH means neither token can pass → 403.
# We present the access token (Step 6), so we require only `allowedClients`.
gw_response = ac.create_gateway(
    name=f"p6-jwt-cedar-gw-{int(time.time())}",
    roleArn=GW_ROLE_ARN,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": discovery_url,
        "allowedClients": [client_id]
    }},
    policyEngineConfiguration={
        "arn": pe_arn,
        "mode": "LOG_ONLY"  # Audit mode — evaluate but don't block
    }
)

gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    elif "FAIL" in gw["status"]:
        print(f"FAILED: {gw.get('statusReasons', [])}")
        break
    time.sleep(5)
print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

In [ ]:
# Step 4: Create Cedar policy — permit at the gateway ARN level
gw_arn = f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{gw_id}"
cedar_statement = f'permit(principal, action, resource == AgentCore::Gateway::"{gw_arn}");'
print(f"Cedar policy:\n  {cedar_statement}")

# Policy names must be unique across the policy engine, so derive one from gw_id
# (create_policy is not idempotent — a static name collides on re-runs).
policy_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"permit_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": cedar_statement}},
    validationMode="IGNORE_ALL_FINDINGS"
)

policy_id = policy_response["policyId"]
print(f"Policy: {policy_id} [{policy_response['status']}]")

for _ in range(12):
    p = ac.get_policy(policyEngineId=pe_id, policyId=policy_id)
    if p["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Policy status: {p['status']}")

In [ ]:
# Step 5: Create KB Target on the JWT + Cedar gateway
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    # Tool description exposed to the agent over MCP — this is what
                    # the LLM reads to decide when to call this KB.
                    "description": (
                        "Search two corporate documents: (1) Octank Financial's 10-K annual "
                        "report — financial statements, asset/liability schedules, exhibits, and "
                        "investor disclosures; and (2) a U.S. tornado background & forecasting "
                        "report — where tornadoes form, annual frequency (~1,200/yr), and NOAA data."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {
                                "numberOfResults": 5
                            }
                        }
                    }
                }]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

target_id = target_response["targetId"]
print(f"Target: {target_id}")

for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")

In [ ]:
# Step 6: Authenticate with Cognito, then retrieve THROUGH the gateway.
# Both auth layers apply here: the JWT proves WHO the caller is (gateway edge),
# and Cedar (LOG_ONLY) evaluates + logs WHAT they may do but does not yet block.
#
# We use the official MCP client over httpx (`pip install mcp`); the Cognito
# access token goes in the Authorization header. dp.retrieve() would bypass the
# gateway entirely (hitting Bedrock directly), so neither JWT nor Cedar would apply.
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.shared.exceptions import McpError

# Cognito login → access token (carries the client_id claim the gateway validates).
auth = cognito.initiate_auth(
    ClientId=client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "testuser", "PASSWORD": "TestPass1"},
)
access_token = auth["AuthenticationResult"]["AccessToken"]
print(f"Access token: {access_token[:40]}...")

# The forbid policy in Step 7 hides the tool from tools/list, so resolve the tool
# name now (while permitted) and reuse it in Step 8.
tool_name = "kb-retrieve___Retrieve"

async with httpx.AsyncClient(headers={"Authorization": f"Bearer {access_token}"}) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session_mcp:
            await session_mcp.initialize()               # MCP handshake (required)

            listing = await session_mcp.list_tools()
            tool_name = next(t.name for t in listing.tools if t.name.split("___")[-1] == "Retrieve")
            print(f"Gateway tool: {tool_name}")

            result = await session_mcp.call_tool(
                name=tool_name,
                arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
            )
            print(f"isError: {result.isError}")
            print("=== Retrieved via gateway (valid JWT, Cedar LOG_ONLY) ===")
            print(result.content[0].text[:800] if result.content else result)

In [ ]:
# Step 7: Switch Cedar to ENFORCE and add a FORBID on this gateway.
# Two things are required for a real block (both verified):
#   1. gateway mode = ENFORCE (LOG_ONLY only logs, never blocks)
#   2. forbid scoped to the gateway RESOURCE (a bare wildcard resource is rejected)
# forbid overrides the earlier permit, so ALL callers are denied on this gateway —
# even a caller with a perfectly valid JWT. This is Layer 2 (authorization) saying
# "no" after Layer 1 (authentication) already said "yes".

# 1. Flip the gateway from LOG_ONLY to ENFORCE (re-send its current config).
gw_info = ac.get_gateway(gatewayIdentifier=gw_id)
ac.update_gateway(
    gatewayIdentifier=gw_id,
    name=gw_info["name"],
    roleArn=gw_info["roleArn"],
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": discovery_url,
        "allowedClients": [client_id]
    }},
    policyEngineConfiguration={"arn": pe_arn, "mode": "ENFORCE"},
)
for _ in range(24):
    if ac.get_gateway(gatewayIdentifier=gw_id)["status"] == "READY":
        break
    time.sleep(5)
print("Gateway mode: ENFORCE")

# 2. Add the forbid policy (unique name per gateway to avoid re-run collisions).
forbid_statement = f'forbid(principal, action, resource == AgentCore::Gateway::"{gw_arn}");'
print(f"Cedar forbid policy:\n  {forbid_statement}")

deny_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"deny_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": forbid_statement}},
    validationMode="IGNORE_ALL_FINDINGS",
)
deny_policy_id = deny_response["policyId"]
for _ in range(12):
    st = ac.get_policy(policyEngineId=pe_id, policyId=deny_policy_id)["status"]
    if st in ("ACTIVE", "CREATE_FAILED"):
        break
    time.sleep(5)
print(f"Deny policy: {deny_policy_id} [{st}]")
time.sleep(15)  # let the decision propagate to the gateway

In [ ]:
# Step 8: Retrieve again with the SAME valid JWT — now BLOCKED by Cedar.
# The JWT is still valid (Layer 1 passes), but ENFORCE + forbid means Cedar
# (Layer 2) denies the tool call: the MCP client raises McpError with JSON-RPC
# code -32002. We catch it INSIDE the session context, where it surfaces directly
# (on context exit it gets wrapped in nested ExceptionGroups, harder to unpack).
# The forbid also hides the tool from tools/list — it now returns [] — so we call
# the tool by the name resolved in Step 6.
async with httpx.AsyncClient(headers={"Authorization": f"Bearer {access_token}"}) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session_mcp:
            await session_mcp.initialize()
            try:
                result = await session_mcp.call_tool(
                    name=tool_name,
                    arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
                )
                print("Unexpected — call was allowed:")
                print(result.content[0].text[:400] if result.content else result)
            except McpError as e:
                if e.error.code == -32002:
                    print("BLOCKED by Cedar policy enforcement (JWT was valid — authorization denied)")
                    print(f"   {e.error.message}")
                else:
                    print(f"Unexpected MCP error [{e.error.code}]: {e.error.message}")

## Multi-Tenant Scenario

In a real deployment, different departments get different KB targets:

```
Gateway (JWT + Cedar)
  ├── Target: engineering-kb  ← Cedar: permit dept=engineering
  ├── Target: marketing-kb   ← Cedar: permit dept=marketing
  └── Target: shared-kb      ← Cedar: permit all
```

Each target can have its own Cedar policy scoping access to specific principals.
JWT claims (like `custom:department`) identify the user; Cedar policies decide
which targets they can invoke.

**Note:** Per-target Cedar policies are a beta feature. Currently, Cedar policies
are evaluated at the gateway level. Target-level policies will enable finer-grained
authorization in future releases.

In [ ]:
# Cleanup — order: policies → target → gateway → policy engine → Cognito → DS → KB
ac.delete_policy(policyEngineId=pe_id, policyId=policy_id)
ac.delete_policy(policyEngineId=pe_id, policyId=deny_policy_id)
ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
time.sleep(3)
ac.delete_gateway(gatewayIdentifier=gw_id)
time.sleep(3)
ac.delete_policy_engine(policyEngineId=pe_id)
cognito.delete_user_pool(UserPoolId=pool_id)
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted: Gateway {gw_id}, PE {pe_id}, Cognito {pool_id}, KB {kb_id}")

## LOG_ONLY vs ENFORCE

| Mode | Behavior | When to use |
|---|---|---|
| `LOG_ONLY` | Cedar evaluates and logs decisions but **does not block** | Testing policies, auditing, gradual rollout |
| `ENFORCE` | Cedar evaluates and **blocks denied requests** | Production after validating in LOG_ONLY |

**Recommended rollout:** Start with `LOG_ONLY`, review CloudWatch logs for unexpected denials,
then switch to `ENFORCE` once policies are validated.

## Per-Target Cedar Policies (Beta Limitation)

Currently, Cedar policies are evaluated at the **gateway level** — they control whether a
principal can invoke the gateway at all. Per-target policies (e.g., permit user X to invoke
target A but not target B) are planned for a future release.

Workaround: Use separate gateways for different access levels, or combine with Lambda
interceptors (Pattern 7) for target-level routing logic.